# 02 — scikit-learn Basics: Training Your First ML Model

**scikit-learn** is the standard Python library for classical machine learning.

You'll learn:
- What a machine learning model actually is
- How to split data into training and test sets
- How to train a **classifier** (predicts a category)
- How to train a **regressor** (predicts a number)
- How to evaluate model performance

The core sklearn pattern is always: **fit → predict → evaluate**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

print('All libraries loaded!')

## Part 1: Classification — "Which category is this?"

We'll classify iris flowers into 3 species based on petal/sepal measurements.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data    # features: 4 measurements per flower
y = iris.target  # labels: 0=setosa, 1=versicolor, 2=virginica

print('Feature shape:', X.shape)   # 150 flowers, 4 features each
print('Label shape:', y.shape)
print('Classes:', iris.target_names)
print('Feature names:', iris.feature_names)

In [ ]:
# Split: 80% train, 20% test
# IMPORTANT: never train and test on the same data — the model would just memorize!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training samples:', len(X_train))
print('Test samples:', len(X_test))

In [ ]:
# Normalize features — KNN is distance-based, so scale matters
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn scale from train, apply
X_test_scaled  = scaler.transform(X_test)       # apply same scale to test

In [ ]:
# Train a K-Nearest Neighbors classifier
# k=5 means: look at the 5 nearest examples and vote on the label
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)  # THE LEARNING HAPPENS HERE

print('Model trained!')

In [ ]:
# Make predictions on the test set
y_pred = knn.predict(X_test_scaled)

# Accuracy: what fraction did we get right?
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.1%}')  # e.g. 96.7%

print()
print('Detailed report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Predict on a new, made-up flower
new_flower = [[5.1, 3.5, 1.4, 0.2]]  # [sepal_length, sepal_width, petal_length, petal_width]
new_flower_scaled = scaler.transform(new_flower)
prediction = knn.predict(new_flower_scaled)
print(f'Predicted species: {iris.target_names[prediction[0]]}')

## Part 2: Regression — "What number is this?"

Regression predicts a continuous value (not a category). We'll predict diabetes progression from medical measurements.

In [ ]:
# Load the diabetes dataset
diabetes = load_diabetes()
X_d = diabetes.data
y_d = diabetes.target  # disease progression score (continuous number)

print('Features shape:', X_d.shape)
print('Target range:', y_d.min(), '—', y_d.max())

In [ ]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42
)

# Train a simple linear regression model
model = LinearRegression()
model.fit(X_train_d, y_train_d)

print('Model trained!')

In [ ]:
y_pred_d = model.predict(X_test_d)

# RMSE: average error in the same units as the target
rmse = np.sqrt(mean_squared_error(y_test_d, y_pred_d))
print(f'RMSE: {rmse:.1f}')
print(f'(Target range: {y_d.min():.0f} — {y_d.max():.0f})')

# Visualize: predicted vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test_d, y_pred_d, alpha=0.6, color='steelblue')
plt.plot([y_d.min(), y_d.max()], [y_d.min(), y_d.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Regression: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

## Exercises — Try These!

1. **Change k in KNN**: Try `n_neighbors=1`, `3`, `10`, `20`. What happens to accuracy?
2. **Different classifier**: Replace `KNeighborsClassifier` with `RandomForestClassifier` (from `sklearn.ensemble`). Does accuracy improve?
3. **Feature importance**: After training a `RandomForestClassifier`, print `model.feature_importances_`. Which feature matters most?

```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
print(rf.score(X_test_scaled, y_test))  # quick accuracy
```
